<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Modern AI: LLMs & Generative AI** · Week 2 · Building with LLM APIs · Lesson 9 of 24

# Structured Output and Tool Use

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

**API key:** this lesson calls an AI model. Add your key with the **key icon (Secrets)** in the left bar, switch on **Notebook access**, and never paste a key into the code.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative).

In [ ]:
# Install the libraries this lesson needs (takes a few seconds).
%pip install -q anthropic

**Learning objective:** Get reliable structured data (JSON) out of an LLM, validate it with Pydantic, and let the model call your own Python functions as tools.

## Text is for people; programs need structure

Your app needs fields it can store and act on — a category, an amount, a phone number — not a paragraph. Two techniques:

1. **Ask for JSON and validate it.**
2. **Use a tool definition (a JSON schema) to force the structure.**

In [ ]:
import importlib.util, os, subprocess, sys

if importlib.util.find_spec("anthropic") is None:          # install once per Colab session
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic"], check=True)

try:
    from google.colab import userdata                     # Colab: key saved in the Secrets panel (key icon)
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    pass                                                    # not in Colab: set ANTHROPIC_API_KEY in your environment
except Exception:
    print("Add a secret called ANTHROPIC_API_KEY in Colab's Secrets panel, then run this cell again.")

import anthropic

client = anthropic.Anthropic()        # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-haiku-5-5"            # fast and low-cost; check docs.claude.com for current model names

def ask(prompt, system=None, max_tokens=500):
    extra = {"system": system} if system else {}
    response = client.messages.create(model=MODEL, max_tokens=max_tokens,
                                      messages=[{"role": "user", "content": prompt}], **extra)
    return response.content[0].text

## Technique 1 — Ask for JSON, then validate

In [ ]:
import json
from pydantic import BaseModel, ValidationError, Field
from typing import Literal

class Ticket(BaseModel):
    category: Literal["payment", "network", "account"]
    urgent: bool
    summary: str = Field(max_length=120)

def extract_ticket(message, attempts=2):
    prompt = (
        "Read the customer message and reply with JSON only, no other text, with keys: "
        '"category" (payment, network or account), "urgent" (true or false), '
        '"summary" (under 120 characters).\n\n'
        f"<message>{message}</message>"
    )
    for attempt in range(attempts):
        raw = ask(prompt, max_tokens=200)
        try:
            return Ticket.model_validate(json.loads(raw))
        except (json.JSONDecodeError, ValidationError) as error:
            print(f"attempt {attempt + 1} failed validation: {type(error).__name__}")
            prompt += "\nYour last reply was not valid. Reply with the JSON object only."
    return None

ticket = extract_ticket("URGENT: money was deducted twice when I paid my rent!")
print(ticket)

**Pydantic** checks every field: the category must be one of the three words, `urgent` must be true or false, the summary must be short. If anything is wrong, we retry once with a reminder, and give up safely (return `None`) instead of crashing.

## Technique 2 — Force the shape with a tool

The API lets you describe a "tool" with a **JSON schema** and require the model to use it. The model then returns arguments that follow the schema. Pydantic can even write the schema for you:

In [ ]:
ticket_tool = {
    "name": "save_ticket",
    "description": "Save a classified customer support ticket.",
    "input_schema": Ticket.model_json_schema(),
}
response = client.messages.create(
    model=MODEL, max_tokens=300,
    tools=[ticket_tool],
    tool_choice={"type": "tool", "name": "save_ticket"},      # must use this tool
    messages=[{"role": "user", "content": "My SIM card is blocked, I cannot receive my salary!"}],
)
tool_call = next(block for block in response.content if block.type == "tool_use")
print(tool_call.input)
print(Ticket.model_validate(tool_call.input))

This is usually more reliable than asking for JSON in words. Still validate: your code, not the model, is responsible for what gets saved.

## Tool use: letting the model call your functions

Tools are not only for structure. You can give the model **real functions** — check a balance, look up an exchange rate, search a database. The model **decides** when a tool is needed and with what arguments; **your code runs it** and sends back the result; the model then writes the final answer.

In [ ]:
# Two "real" functions (here with practice data)
BALANCES = {"0788123456": 45_300, "0722000111": 1_250}
RATES_TO_RWF = {"USD": 1_400, "EUR": 1_500, "KES": 10.8}       # practice rates, not live data

def get_balance(phone: str) -> str:
    return f"{BALANCES.get(phone, 0):,} RWF" if phone in BALANCES else "unknown number"

def convert_to_rwf(amount: float, currency: str) -> str:
    rate = RATES_TO_RWF.get(currency.upper())
    return f"{amount * rate:,.0f} RWF" if rate else "unknown currency"

TOOLS = [
    {"name": "get_balance", "description": "Get the mobile-money balance of a phone number.",
     "input_schema": {"type": "object", "properties": {"phone": {"type": "string"}}, "required": ["phone"]}},
    {"name": "convert_to_rwf", "description": "Convert an amount of a foreign currency to Rwandan francs.",
     "input_schema": {"type": "object",
                      "properties": {"amount": {"type": "number"}, "currency": {"type": "string", "enum": ["USD", "EUR", "KES"]}},
                      "required": ["amount", "currency"]}},
]
FUNCTIONS = {"get_balance": get_balance, "convert_to_rwf": convert_to_rwf}

## The tool loop

In [ ]:
def run_with_tools(question, max_rounds=5):
    messages = [{"role": "user", "content": question}]
    for _ in range(max_rounds):
        response = client.messages.create(model=MODEL, max_tokens=500, tools=TOOLS, messages=messages)
        if response.stop_reason != "tool_use":
            return "".join(b.text for b in response.content if b.type == "text")
        messages.append({"role": "assistant", "content": response.content})
        results = []
        for block in response.content:
            if block.type == "tool_use":
                output = FUNCTIONS[block.name](**block.input)        # YOUR code runs the tool
                print(f"  tool {block.name}({block.input}) -> {output}")
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": output})
        messages.append({"role": "user", "content": results})
    return "Stopped: too many tool rounds."

print(run_with_tools("What is the balance of 0788123456?"))

Step by step: the model replies with `stop_reason == "tool_use"` and the tool name and arguments; your code runs the function; you send a `tool_result` back; the model writes a normal answer using the result. `max_rounds` stops endless loops.

## Safety rules for tools

- The model only **asks**; **your code decides** whether to run the tool. Check arguments (is this phone number allowed for this user?).
- Give read-only tools first. Anything that moves money or deletes data needs a **human confirmation** step.
- Log every tool call.

## Key terms

- **Structured output** — data in a fixed format such as JSON.
- **JSON schema** — a description of the fields and types allowed.
- **Pydantic** — a Python library that validates data against a model.
- **Tool use (function calling)** — the model requests a function call; your code runs it.
- **Tool result** — the output you send back for the model to use.

**Takeaway:** Validate every structured output with code, use tool schemas to force the shape, and remember that in tool use the model only asks — your program decides and acts.

## Try it yourself

1. Add a field `amount_rwf: int | None` to `Ticket` and test it on messages with and without an amount.
2. Ask "How much is 50 US dollars in Rwandan francs?" with `run_with_tools`. Which tool is used?
3. Add a tool `report_fraud(phone, description)` that only **prints** a report. When should it require a person to confirm?
4. Make `get_balance` refuse numbers that do not belong to the logged-in user. Where does that check belong — in the prompt or in the code?

---
Finished? Go back to [Modern AI: LLMs & Generative AI on WAMUGII](https://wamugii.kento.rw/courses/modern-ai-llms-generative), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*